In [5]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent

# .env 파일에서 환경 변수 로드
load_dotenv()

# 모델 선언
model = init_chat_model("gpt-4o-mini")

In [6]:
from typing import List, Dict
from langchain.tools import tool

@tool
def send_email_tool(to: str, subject: str, body: str) -> str:
    """지정한 주소로 이메일을 보내는 도구(프로토타입)."""
    return f"Email sent. to={to}, subject={subject}"

@tool
def read_email_tool(limit: int = 3) -> List[Dict[str, str]]:
    """최근 받은 이메일을 조회하는 도구(프로토타입)."""
    return [{"from": "hr@example.com", "subject": "정책 안내", "body": "..."}][:limit]

In [7]:
from langchain.agents.middleware import LLMToolEmulator
from langchain.agents.middleware import TodoListMiddleware

agent = create_agent(
    model=model,
    tools=[send_email_tool, read_email_tool],
    # tool emulator: 실제 도구 호출 대신 LLM으로 시뮬레이션, 즉, 도구가 만든 결과를 모킹하는것. gemini-2.5-flash 모델 사용
    middleware=[LLMToolEmulator(model="gpt-4o-mini"), TodoListMiddleware()],
    )

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "온 메일 다 확인한 뒤 요약해 보고해. 그 다음 답장 작성해 회신 보내줘. 마지막으로 어떻게 보냈는지 보고해.",
            }
        ]
    }
)
print("=== 이메일 읽고 쓰기 결과 ===")
print(result)
print(result['messages'][-1].content)



=== 이메일 읽고 쓰기 결과 ===
{'messages': [HumanMessage(content='온 메일 다 확인한 뒤 요약해 보고해. 그 다음 답장 작성해 회신 보내줘. 마지막으로 어떻게 보냈는지 보고해.', additional_kwargs={}, response_metadata={}, id='b62f80c2-41b6-4eb2-8c49-514cde02971b'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 1201, 'total_tokens': 1216, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_668125ce6a', 'id': 'chatcmpl-DLfa2UMX6G85a1vyKFK58xosJxtQE', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d0e0a-fbd7-7b43-abd0-1ef73480c706-0', tool_calls=[{'name': 'read_email_tool', 'args': {'limit': 3}, 'id': 'call_TYarDdsUeZi0zubuE8RIvVGB', 'type': 'tool_call'}], invalid_tool

**`TodoListMiddleware`**(또는 유사한 Todo 관리 기능)가 작동하고 있음을 알 수 있는 결정적인 증거는 **`write_todos`라는 도구(tool)가 호출되는 과정**입니다.

구체적으로 다음 세 부분을 확인하시면 됩니다.

### 1. `AIMessage`의 `tool_calls` 목록
로그 중간중간에 AI가 스스로 할 일을 정리하기 위해 `write_todos`를 호출하는 것을 볼 수 있습니다.

*   **첫 번째 호출 (할 일 등록):**
    ```json
    "tool_calls": [{
        "name": "write_todos", 
        "args": {"todos": [{"content": "모든 이메일을 요약한다.", "status": "in_progress"}]},
        ...
    }]
    ```
    AI가 "이제 이메일 요약을 시작할 거야"라고 Todo 리스트에 상태를 업데이트하는 장면입니다.

### 2. `ToolMessage`의 응답
AI가 `write_todos`를 호출한 직후, 시스템(Middleware)이 그 결과를 받아 저장했다는 응답이 옵니다.
```json
ToolMessage(content="{'todos': [{'content': '모든 이메일을 요약한다.', 'status': 'in_progress'}]}", name='write_todos', ...)
```

### 3. 지속적인 상태 업데이트 (진행 상황 관리)
단순히 한 번 호출하고 끝나는 게 아니라, 작업이 진행됨에 따라 **상태(`status`)를 변경**하며 계속 호출합니다.

*   **상태 변경 (completed):**
    ```json
    "args": {
        "todos": [
            {"content": "모든 이메일을 요약한다.", "status": "completed"},
            {"content": "답장 초안을 작성한다.", "status": "pending"}
        ]
    }
    ```
    요약이 끝나자 "완료(completed)"로 바꾸고 다음 할 일인 "답장 초안"을 추가하는 것을 볼 수 있습니다.

### 4. 최종 보고 전 마지막 업데이트
모든 작업이 끝나기 직전에도 Todo 리스트를 최종 업데이트합니다.
```json
"args": {
    "todos": [
        {"content": "모든 이메일을 요약한다.", "status": "completed"},
        {"content": "답장 초안을 작성한다.", "status": "completed"},
        {"content": "답장 보낸 내용을 보고한다.", "status": "completed"}
    ]
}
```

### 요약
로그에서 **`write_todos`라는 이름의 도구가 여러 번 호출**되고, 그 인자(`args`)로 **`content`와 `status`가 포함된 리스트**가 전달되는 과정을 통해 `TodoListMiddleware`가 배후에서 AI의 작업 순서를 관리하고 있음을 명확히 알 수 있습니다. 

AI는 이 미들웨어 덕분에 사용자의 복잡한 요청(요약 -> 답장 -> 보고)을 단계별로 잊지 않고 수행할 수 있었던 것입니다.

------

`TodoListMiddleware`는 질문의 복잡도와 상관없이 **항상 호출되지만, AI가 실제로 할 일을 리스트로 만들지(Write) 여부는 AI의 판단**에 달려 있습니다.

구체적으로 어떻게 동작하는지 설명해 드릴게요.

### 1. 미들웨어는 항상 "대기 중"
`middleware=[..., TodoListMiddleware()]`라고 설정되어 있기 때문에, 에이전트가 실행될 때마다 이 미들웨어는 AI의 모든 입출력을 지켜보고 있습니다.

### 2. AI의 판단 (간단한 질문일 때)
만약 질문이 "지금 몇 시야?" 또는 "안녕?" 같은 간단한 것이라면:
- AI는 굳이 할 일을 단계별로 나눌 필요가 없다고 판단합니다.
- 따라서 `write_todos`라는 도구를 호출하지 않습니다.
- 이 경우 미들웨어는 아무런 작업도 하지 않고 그냥 통과시킵니다.

### 3. AI의 판단 (복잡한 질문일 때)
질문이 "메일 확인하고, 요약하고, 답장 보내줘"처럼 여러 단계라면:
- AI는 "이건 한 번에 하기 힘들겠는데? 단계를 나눠야겠다"라고 생각합니다.
- AI가 스스로 `write_todos` 도구를 호출하여 할 일을 등록합니다.
- 이때부터 미들웨어가 본격적으로 개입하여 할 일 목록을 관리하기 시작합니다.

### 4. 왜 이렇게 만드나요?
모든 질문에 억지로 Todo 리스트를 만들게 하면 오히려 속도가 느려지고 비용이 많이 듭니다. 
- **간단한 질문:** 바로 답변 (Todo 리스트 생성 안 함)
- **복잡한 질문:** Todo 리스트를 만들어 체계적으로 수행

### 요약
- **미들웨어 자체는 항상 호출됩니다.** (감시 모드)
- 하지만 **Todo 리스트를 실제로 작성할지 말지는 AI가 질문의 복잡도를 보고 결정**합니다.
- 따라서 아주 간단한 질문에는 로그에 `write_todos` 호출이 나타나지 않을 것입니다.